In [1]:
%load_ext autoreload
%autoreload 2

import pandas as pd
from src.data_loading import TARGET_COL, ID_COL, PROCESSED_DATA_PATH, load_data_with_secondary_tables
from src.evaluation import get_feature_importance
from src.tuning import (
    tune_lgbm_fast,
    confirm_lgbm_params_cv,
    confirm_lgbm_candidates_cv,
    save_lgbm_params,
    load_lgbm_params,
)

application_train, application_test = load_data_with_secondary_tables()
X = application_train[[c for c in application_train.columns if c not in [TARGET_COL, ID_COL]]]
y = application_train[TARGET_COL]

X.shape

(307511, 568)

In [2]:
N_TRIALS = 30
TOP_N = 3
TUNING_SAMPLE_SIZE = 75000
CONFIRM_N_SPLIT = 5

In [3]:
tuning_results = tune_lgbm_fast(
    X,
    y,
    n_trials=N_TRIALS,
    top_n=TOP_N,
    sample_size=TUNING_SAMPLE_SIZE,
    random_state=42,
)

tuning_results["top_trials"]

,candidate,trial_number,fast_auc,best_iteration
0,1,21,0.770249,180
1,2,10,0.769696,195
2,3,4,0.768896,299


In [4]:
candidate_cv_table, candidate_cv_results = confirm_lgbm_candidates_cv(
    X,
    y,
    tuning_results["top_params"],
    n_split=CONFIRM_N_SPLIT,
    random_state=42,
)

candidate_summary = tuning_results["top_trials"].merge(
    candidate_cv_table,
    on="candidate",
    how="left",
).sort_values("oof_auc", ascending=False)

candidate_summary

,candidate,trial_number,fast_auc,best_iteration,mean_auc,std_auc,oof_auc
1,2,10,0.769696,195,0.787833,0.003364,0.787809
0,1,21,0.770249,180,0.786792,0.004272,0.786766
2,3,4,0.768896,299,0.785604,0.003870,0.785585


In [5]:
best_candidate = int(candidate_summary.iloc[0]["candidate"])
best_lgbm_params = tuning_results["top_params"][best_candidate - 1]

best_params_path = save_lgbm_params(best_lgbm_params, "best_lgbm_params.json")
top_params_path = save_lgbm_params(tuning_results["top_params"], "top_lgbm_params.json")
candidate_summary_path = PROCESSED_DATA_PATH / "lgbm_tuning_candidate_summary.csv"
candidate_summary.to_csv(candidate_summary_path, index=False)

{
    "best_params_path": best_params_path,
    "top_params_path": top_params_path,
    "candidate_summary_path": candidate_summary_path,
    "best_lgbm_params": best_lgbm_params,
}

{'best_params_path': PosixPath('/Users/ryanyao/Quant Projects/Home_Credit/data/processed/best_lgbm_params.json'),
 'top_params_path': PosixPath('/Users/ryanyao/Quant Projects/Home_Credit/data/processed/top_lgbm_params.json'),
 'candidate_summary_path': PosixPath('/Users/ryanyao/Quant Projects/Home_Credit/data/processed/lgbm_tuning_candidate_summary.csv'),
 'best_lgbm_params': {'num_leaves': 89,
  'learning_rate': 0.05443779438685737,
  'min_child_samples': 293,
  'subsample': 0.972916136764715,
  'colsample_bytree': 0.775248732733177,
  'reg_alpha': 1.475649304728376,
  'reg_lambda': 4.3444691085504035,
  'max_depth': -1,
  'subsample_freq': 1,
  'n_estimators': 195}}